In [1]:
import cv2
import numpy as np
import zipfile
from pathlib import Path
from google.colab import files

IMAGE_WIDTH = 640
IMAGE_HEIGHT = 400
PATTERN_SIZE = (9, 6)
SQUARE_SIZE_MM = 25.0
OUTPUT_ZIP = Path("/content/test_dataset.zip")

K_left = np.array([
    [430.626953125, 0.0, 320.33416748046875],
    [0.0, 430.3931884765625, 201.5816192626953],
    [0.0, 0.0, 1.0]
], dtype=np.float64)
K_right = np.array([
    [430.0867614746094, 0.0, 318.8838806152344],
    [0.0, 429.73675537109375, 204.40399169921875],
    [0.0, 0.0, 1.0]
], dtype=np.float64)
D_left = np.array([-0.035, 0.008, 0.0003, -0.0002, 0.001], dtype=np.float64)
D_right = np.array([-0.032, 0.006, 0.0002, -0.0002, 0.001], dtype=np.float64)
R_lr = np.array([
    [0.9998782277107239, 0.010038220323622227, 0.011950093321502209],
    [-0.010067159633636, 0.9999465346336365, 0.0023639767896384],
    [-0.01192572433501482, -0.0024839923717081547, 0.9999257922172546]
], dtype=np.float64)
T_lr_mm = np.array([-75.20867, 0.562431, 0.602984], dtype=np.float64).reshape(3, 1)

cols, rows = PATTERN_SIZE
square = SQUARE_SIZE_MM
center = np.array([(cols - 1) * square / 2.0, (rows - 1) * square / 2.0, 0.0])
rng = np.random.default_rng(20261030)

def project_square(image, R_cam, t_cam, K, D, c, r, color):
    points = np.array([
        [(c - 1) * square, (r - 1) * square, 0.0],
        [c * square, (r - 1) * square, 0.0],
        [c * square, r * square, 0.0],
        [(c - 1) * square, r * square, 0.0]
    ], dtype=np.float32)
    rvec, _ = cv2.Rodrigues(R_cam)
    projected, _ = cv2.projectPoints(points, rvec, t_cam, K, D)
    polygon = np.round(projected.reshape(-1, 2)).astype(np.int32)
    cv2.fillConvexPoly(image, polygon, int(color), lineType=cv2.LINE_AA)
    return polygon

tmp_dir = Path("/content/test_gen")
left_dir = tmp_dir / "CAM_B"
right_dir = tmp_dir / "CAM_C"
left_dir.mkdir(parents=True, exist_ok=True)
right_dir.mkdir(parents=True, exist_ok=True)

made = 0
attempts = 0
while made < 36 and attempts < 250:
    attempts += 1
    rotation_vector = np.deg2rad(rng.uniform([-22, -25, -32], [22, 25, 32]))
    R_board, _ = cv2.Rodrigues(rotation_vector)
    z = float(rng.uniform(560.0, 820.0))
    center_t = np.array([
        rng.uniform(-145.0, 145.0),
        rng.uniform(-85.0, 85.0),
        z
    ], dtype=np.float64).reshape(3, 1)
    t_left = center_t - R_board @ center.reshape(3, 1)
    R_right = R_lr @ R_board
    t_right = R_lr @ t_left + T_lr_mm

    left_image = np.full((IMAGE_HEIGHT, IMAGE_WIDTH), 180, dtype=np.uint8)
    right_image = np.full((IMAGE_HEIGHT, IMAGE_WIDTH), 180, dtype=np.uint8)
    left_polygons, right_polygons = [], []
    for row in range(rows + 1):
        for col in range(cols + 1):
            shade = 245 if (row + col) % 2 == 0 else 15
            left_polygons.append(project_square(left_image, R_board, t_left, K_left, D_left, col, row, shade))
            right_polygons.append(project_square(right_image, R_right, t_right, K_right, D_right, col, row, shade))

    all_points = np.vstack(left_polygons + right_polygons)
    if (all_points[:, 0].min() < 5 or all_points[:, 0].max() >= IMAGE_WIDTH - 5
            or all_points[:, 1].min() < 5 or all_points[:, 1].max() >= IMAGE_HEIGHT - 5):
        continue

    gain = float(rng.uniform(0.92, 1.08))
    bias = float(rng.uniform(-5.0, 5.0))
    for image in (left_image, right_image):
        adjusted = np.clip(image.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)
        noise = rng.normal(0, 0.6, adjusted.shape).astype(np.float32)
        adjusted = np.clip(adjusted.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        if rng.random() < 0.25:
            adjusted = cv2.GaussianBlur(adjusted, (3, 3), 0.35)
        if image is left_image:
            left_image = adjusted
        else:
            right_image = adjusted

    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
    found_left, _ = cv2.findChessboardCorners(left_image, PATTERN_SIZE, flags)
    found_right, _ = cv2.findChessboardCorners(right_image, PATTERN_SIZE, flags)
    if not (found_left and found_right):
        continue

    made += 1
    cv2.imwrite(str(left_dir / f"{made:03d}.png"), left_image)
    cv2.imwrite(str(right_dir / f"{made:03d}.png"), right_image)

print(f"Made {made} pairs")

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(left_dir.glob("*.png")):
        z.write(p, f"CAM_B/{p.name}")
    for p in sorted(right_dir.glob("*.png")):
        z.write(p, f"CAM_C/{p.name}")

print("ZIP ready:", OUTPUT_ZIP.name)
files.download(str(OUTPUT_ZIP))

Made 36 pairs
ZIP ready: test_dataset.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>